[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sampriti-sr/MLP_Course/blob/kshitij-variant/docs/toolkit/sklearn-api.ipynb)

# scikit-learn: The Estimator API

<div class="admonition info">
<p class="admonition-title">What you will be able to do after this page</p>
<ul>
<li>Use the fit / transform / predict pattern and recognise learned attributes.</li>
<li>Split data repeatably and stratified, and fit only on the training set.</li>
<li>Judge a model against a baseline, and spot the accuracy trap.</li>
<li>Read predict_proba correctly using classes_.</li>
</ul>
</div>

**Prerequisites:** <a href="../seaborn/">Seaborn: Statistical Charts</a>. **Dataset:** Credit Card Default (Taiwan, 2005). **Time:** about 40 minutes.

In [1]:
try:
    import mlp_data
except ImportError:  # e.g. a fresh Colab runtime
    %pip install -q "mlp-course @ git+https://github.com/sampriti-sr/MLP_Course@kshitij-variant"
    import mlp_data

import time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mlp_data import load_credit, style

style.apply()
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 14)
pd.set_option("display.float_format", "{:,.3f}".format)

## The situation

scikit-learn's great design idea is that **every tool looks the same**. Whether you are scaling a column, fitting a regression or clustering, you call the same few methods in the same way. Learn the pattern once and a hundred classes become familiar.

Our data: 30,000 credit-card customers in Taiwan (2005), 23 features describing their credit limit, repayment history, bills and payments, and one target: did they **default** next month? Only about one customer in five did, which makes it a good lesson in why accuracy is a poor guide.

In [2]:
credit = load_credit()
target = "default payment next month"
y = credit[target]
X = credit.drop(columns=["ID", target])
print(f"{X.shape[0]:,} customers, {X.shape[1]} features; default rate {y.mean():.1%}")
X.iloc[:3, :8]

30,000 customers, 23 features; default rate 22.1%


,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3
0,20000,2,2,1,24,2,2,-1
1,120000,2,2,2,26,-1,2,0
2,90000,2,2,2,34,0,0,0


The most important feature is `PAY_0`, the customer's repayment status last month (−2 and −1 mean paid early or on time; 1 means one month late, 2 means two months late, and so on). Before any model, look at how default depends on it:

In [3]:
by_status = y.groupby(X["PAY_0"]).agg(customers="size", default_rate="mean")
by_status.assign(default_rate=by_status["default_rate"].map("{:.0%}".format))

,customers,default_rate
PAY_0,,
-2,2759,13%
-1,5686,17%
0,14737,13%
1,3688,34%
2,2667,69%
3,322,76%
4,76,68%
5,26,50%
6,11,55%


The relationship is strongly **non-linear**: customers who are on time (−2, −1, 0) default about 13% to 17% of the time, but one month late is 34%, two months late is 69% and three months late 76%. A straight line through those numbers would be a poor description, a point we return to on the preprocessing page.

## 1. Three roles, one pattern

Every scikit-learn object plays one or more of three roles:

| Role | Method | What it does | Example |
|---|---|---|---|
| **Estimator** | `fit(X, y)` | learns something from data | every model and transformer |
| **Transformer** | `transform(X)` | turns data into new data | `StandardScaler`, `OneHotEncoder` |
| **Predictor** | `predict(X)`, `predict_proba(X)`, `score(X, y)` | makes predictions | `LogisticRegression`, `RandomForestClassifier` |

What an estimator *learns* is stored in attributes that **end with an underscore**, so you can always tell configuration (set by you) from learned state (set by `fit`):

In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

scaler = StandardScaler()                               # configuration only: nothing learned yet
print("before fit, has mean_?", hasattr(scaler, "mean_"))
scaler.fit(X)                                           # learn each column's mean and standard deviation
print("after fit, mean of LIMIT_BAL (learned):", round(scaler.mean_[0]), "| scale:", round(scaler.scale_[0]))
Z = scaler.transform(X)                                 # apply what was learned
print("transformed column 0 now has mean", round(Z[:, 0].mean(), 6), "and std", round(Z[:, 0].std(), 3))

before fit, has mean_? False
after fit, mean of LIMIT_BAL (learned): 167484 | scale: 129745
transformed column 0 now has mean -0.0 and std 1.0


`fit_transform` does both steps in one call. Configuration lives in `get_params()`, and `clone` makes a fresh, **unfitted** copy with the same settings, which is how cross-validation gets a clean model for each fold:

In [5]:
model = LogisticRegression(C=0.5, max_iter=500)
print(model.get_params()["C"], model.get_params()["max_iter"])
from sklearn.base import clone
fresh = clone(model)
print("clone is unfitted:", not hasattr(fresh, "coef_"))

from sklearn.exceptions import NotFittedError
try:
    model.predict(X)
except NotFittedError as e:
    print("predict before fit ->", type(e).__name__)

0.5 500
clone is unfitted: True
predict before fit -> NotFittedError


## 2. Always evaluate on data the model has not seen

Split the data into a **training set** (the model learns from it) and a **test set** (we judge it on it). Two details matter: `random_state` makes the split repeatable, and `stratify=y` keeps the default rate the same in both parts, which matters when one class is rare.

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
print(f"train {len(X_train):,} rows (default rate {y_train.mean():.2%}); test {len(X_test):,} rows (default rate {y_test.mean():.2%})")

# How much would the test default rate wobble without stratify? Repeat the split with 200 different seeds.
rates = [train_test_split(X, y, test_size=0.05, random_state=s)[3].mean() for s in range(200)]
strat = [train_test_split(X, y, test_size=0.05, random_state=s, stratify=y)[3].mean() for s in range(200)]
print(f"small test sets: default rate varies by about +/-{np.std(rates):.3f} without stratify, +/-{np.std(strat):.4f} with it")
assert np.std(strat) < np.std(rates) / 5

train 22,500 rows (default rate 22.12%); test 7,500 rows (default rate 22.12%)


small test sets: default rate varies by about +/-0.010 without stratify, +/-0.0000 with it


Stratifying pins the class share almost exactly, while an unstratified split lets it wander by a percentage point or so (more with smaller sets). With a rare class that wobble can decide whether a metric looks good.

## 3. A baseline first, and the accuracy trap

Before trusting any model, ask what a model that learns *nothing* scores. `DummyClassifier` is that model: it always predicts the most common class.

In [7]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

dummy = DummyClassifier(strategy="prior").fit(X_train, y_train)
from sklearn.pipeline import make_pipeline
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train, y_train)

rows = {}
for name, m in [("always predict 'no default'", dummy), ("logistic regression", model)]:
    rows[name] = {"accuracy": accuracy_score(y_test, m.predict(X_test)), "AUC": roc_auc_score(y_test, m.predict_proba(X_test)[:, 1])}
pd.DataFrame(rows).T

,accuracy,AUC
always predict 'no default',0.779,0.500
logistic regression,0.811,0.711


**This is the accuracy trap.** The model that predicts "no default" for everyone is right 78% of the time and finds *no one* who will default (its AUC is 0.5, a coin flip). The logistic regression is only about three points more accurate, yet its AUC (about 0.71) shows it ranks customers far better than chance. Judge a model against a baseline, and use a metric that suits the question; the evaluation page goes further.

## 4. What `predict` and `predict_proba` return

In [8]:
proba = model.predict_proba(X_test)
print("classes_:", model.classes_, "| proba shape:", proba.shape, "| each row sums to 1:", bool(np.allclose(proba.sum(axis=1), 1)))
print("column 1 is the probability of class 1 (default): first five customers ->", proba[:5, 1].round(3))
print("predict() is just 'proba above 0.5':", bool(((proba[:, 1] > 0.5).astype(int) == model.predict(X_test)).all()))

classes_: [0 1] | proba shape: (7500, 2) | each row sums to 1: True
column 1 is the probability of class 1 (default): first five customers -> [0.194 0.045 0.168 0.198 0.117]
predict() is just 'proba above 0.5': True


`predict_proba` returns one column per class, **in the order of `classes_`**, so column 1 is the probability of class 1. `predict` simply thresholds it at 0.5, and 0.5 is a *choice*, not a law; for a rare class a different threshold is often better (evaluation page).

## Pitfalls at a glance

| Surprise | Cause | Fix |
|---|---|---|
| `NotFittedError` | called `predict`/`transform` before `fit` | fit first (or you reused an unfitted clone) |
| 78% accuracy looks great | class imbalance; the baseline scores the same | compare with `DummyClassifier`; use AUC/precision/recall |
| Test metrics fluctuate between runs | no `random_state`, no `stratify` | set both for repeatable, balanced splits |
| Wrong class probability used | assumed column 0 | check `model.classes_` |
| Scaler "learned" from the test set | called `fit` on the whole data | fit on **training** data only, then `transform` the test set |

## Check yourself

<details class="question">
<summary>1. How can you tell, from its attributes, whether an estimator has been fitted?</summary>

Fitted estimators have attributes ending in an underscore (`coef_`, `mean_`, `classes_`), created by `fit`. An unfitted one lacks them. Underscore means "learned from data", no underscore means "configured by you".

</details>

<details class="question">
<summary>2. Why <code>fit</code> the scaler on the training set only, and just <code>transform</code> the test set?</summary>

The test set stands in for future data you have not seen. If the scaler learned its mean and standard deviation from the test rows too, information about the test set would leak into the preparation. Fit on train, then apply the same learned transformation to test, exactly as you will on new data in production.

</details>

<details class="question">
<summary>3. A model scores 93% accuracy on data where 92% of cases are negative. Is that good?</summary>

Almost certainly not: always predicting negative scores 92%. The model is one point better than doing nothing. Compare against the baseline, and use a metric (AUC, precision and recall, average precision) that is not dominated by the majority class.

</details>

## Exercises

<details class="question">
<summary>Refit the logistic regression with a much stronger penalty (<code>C=0.001</code>). Predict whether the coefficients grow or shrink, and check with <code>coef_</code>.</summary>

```python
from sklearn.pipeline import make_pipeline
for C in (1.0, 0.001):
    m = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=1000)).fit(X_train, y_train)
    print(f"C={C}: largest |coefficient| = {np.abs(m[-1].coef_).max():.3f}")
```

`C` is the **inverse** of the regularisation strength, so a smaller `C` penalises large coefficients more and shrinks them toward zero (the largest drops from about 0.65 to about 0.46). (`m[-1]` picks the last step of the pipeline, the logistic regression, whose `coef_` we want.)

</details>

<details class="question">
<summary>Show that <code>predict_proba</code> columns follow <code>classes_</code> by fitting on labels coded 'no' and 'yes' instead of 0 and 1.</summary>

```python
labels = y_train.map({0: "no", 1: "yes"})
m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train, labels)
print(m.classes_)                                  # ['no' 'yes']: sorted alphabetically
print(m.predict_proba(X_test)[:3].round(3))        # column 0 = 'no', column 1 = 'yes'
```

`classes_` is sorted, so with string labels the order can differ from your intuition. Always read the column for your class off `classes_` rather than assuming.

</details>

## Recap

- Every object follows **fit / transform / predict**; learned state lives in attributes ending in `_`.
- `clone` gives a fresh unfitted copy; `get_params` / `set_params` read and change configuration.
- Split with `random_state` and `stratify`, **fit on train only**, and evaluate on the test set.
- Always compare with a **baseline** (`DummyClassifier`): on imbalanced data, accuracy is a trap.
- `predict_proba` columns follow `classes_`; `predict` thresholds at 0.5, which is adjustable.

**Next:** *Preprocessing*, getting features into the shape each model needs.